# Building Trust in Code Reviews using Evals


## Introduction

Code reviews are one of the most important places where an AI coding system has to be both useful and trustworthy. A model that writes code can still create a lot of damage if its review behavior is noisy, speculative, or blind to important failures. So the question is not just "can the model produce review comments?" It is "how do we know whether those reviews are actually good?" 

Reliable eval systems build trust in code review automation, and allow teams to experiment with models, reasoning levels, system prompts, `AGENTS.md` files, and context engineering to optimize for performance (and cost). 

In this guide we will build three evals (with the OpenAI [Evals API](https://developers.openai.com/api/reference/resources/evals)) of increasing complexity, each demonstrating concepts to improve the accuracy or utility of the system. Refer to [this guide](https://developers.openai.com/cookbook/examples/evaluation/getting_started_with_openai_evals) if you are new to the concept.

We will use the open-source [openai/codex](https://github.com/openai/codex) repository as our data source. If you would like to follow along, the runnable reference implementation is available [here](evals/codereview_evals). 

## Foundations

To construct a useful eval for code review, we need four pieces:
- a golden dataset
- a code reviewer model, the system under test
- grader logic, the LLM-as-a-judge
- score aggregation


### The Golden Dataset

A golden dataset is a trusted set of examples that you use to measure model behavior. In general, eval datasets are best when they are representative - covering typical use-cases, edge cases, and "adversarial cases" (deliberately difficult situations designed to cause failures). Larger datasets are not necessarily better. The best practice is to start with a smaller set of curated examples, and add to it over time.

In this guide, we will use closed pull requests in [`openai/codex`](https://github.com/openai/codex) gathered with the [GitHub CLI](https://cli.github.com). For each PR, we collect:
1. metadata
2. changed files
3. unified diff
4. comments
5. outcome (`merged` or `closed-unmerged`)

That gives us a pragmatic benchmark dataset with minimal time spent curating. The comments are not treated as perfect labels, but they are still valuable reference evidence. The merged state is not a perfect proxy for review quality, but it is a useful dimension for interpreting results.

### System Under Test

The system under test is the code reviewer model. In the reference implementation, the reviewer takes a PR from the dataset and produces a code review. The eval system should provide context that closely resembles the real-world deployment of a code review agent, which could include:
- review context (from the dataset)
- system prompt
- `AGENTS.md` 
- Tools (system tools, MCP Servers, skills)

### Graders

In general, [graders](https://developers.openai.com/api/docs/guides/graders) are the *scoring rules* used to compare output from the model-under-test to the golden dataset. They can be both deterministic (ex: `text_similarity`) and non-deterministic (LLM-as-a-Judge). LLM-as-a-Judge is a good choice here because code review outputs are rich and open-ended. The grader model will score each generataed code review along the following axes:
- correctness
- usefulness
- noise
- overall pass

### Score Aggregation

The Evals API handles per-example grading, but we still need a way to summarize the run. We'll start with this summary aggregation:

```text
correctness_rate = passed_correctness_examples / total_examples
usefulness_rate = passed_usefulness_examples / total_examples
noise_rate = passed_noise_examples / total_examples
overall_pass_rate = passed_overall_examples / total_examples
```

We can also compute subgroup slices such as merged vs closed-unmerged pull requests. Those slices are not the objective of the eval, but they are often the fastest way to spot distribution shifts or hidden failure modes.


## Harness Setup

In the reference implementation, we've provided an Eval CLI to easily run the examples. Install it with:

```bash
cd examples/evals/codereview_evals
python -m venv .venv
source .venv/bin/activate
pip install -e .

# Test your installation
evalcr --help
```

## Level 1: Basic Benchmark Eval

In the simple benchmark eval, the reviewer model generates a review for each PR, and the grader evaluates that review against the PR context.

The evaluation loop looks like this:

```mermaid
flowchart LR
    A[Prepared PR Record] --> B[Reviewer Generates Review]
    B --> C[Graders Score Review]
    C -. Repeat for the next PR .-> A
    classDef reviewer fill:#FEF3C7,stroke:#D97706,color:#78350F,stroke-width:1.5px;
    classDef grader fill:#DCFCE7,stroke:#16A34A,color:#14532D,stroke-width:1.5px;
    class B reviewer;
    class C grader;
```

### 1. Create the Dataset

First, fetch PRs from your repository and cache them locally:
```bash
evalcr fetch-prs --repo openai/codex --limit 50
```

Then, we will prepare the raw dataset for the Eval API:
```bash
evalcr prepare-dataset --level 1 --cache-key openai_codex
```

Each Level 1 row contains a prepared pull request record with:
- stable PR metadata
- merged state
- a deterministic changed-file summary
- a truncated diff excerpt
- normalized historical comments

This dataset is still close to the raw data from GitHub. That is intentional; we'll first observe a straightforward benchmark before we start improving the system.

### 2. Define the Eval

NOTE: if you want to start the eval now while you read the following sections, use this command (which wraps and executes the next few sections):
```bash
evalcr run-evals --level 1 --cache-key openai_codex
```

First, we'll create and configure an eval using the OpenAI Python SDK. The `data_source_config` and `testing_criteria` are shortened here for legibility; the complete implementation is in [evals_api.py](evals/codereview_evals/codereview_evals/evals_api.py), and a full overview of this harness can be found in the [README.md](evals/codereview_evals/README.md)

```python
from openai import OpenAI

client = OpenAI()

eval_obj = client.evals.create(
    name="code-review-evals-level-1",
    data_source_config={
        "type": "custom",
        "item_schema": {"type": "object", "properties": {...}, "required": [...]},
        "include_sample_schema": True,
    },
    testing_criteria=[
        {"type": "label_model", "name": "Correctness", ...},
        {"type": "label_model", "name": "Usefulness", ...},
        {"type": "label_model", "name": "Noise", ...},
        {"type": "label_model", "name": "Overall pass", ...},
    ],
)
```

### 3. Run the Reviewer Against the Dataset

Now we upload the dataset and run a reviewer prompt over it. Note that in this example we're using `gpt-5.3-codex` as the review model.

```python
uploaded = client.files.create(
    file=open("data/prepared/openai_codex/level_1/benchmark.jsonl", "rb"),
    purpose="evals",
)

run = client.evals.runs.create(
    eval_obj.id,
    name="level-1-demo",
    data_source={
        "type": "responses",
        "model": "gpt-5.3-codex",
        "input_messages": {
            "type": "template",
            "template": [
                {"role": "developer", "content": "...reviewer instructions..."},
                {"role": "user", "content": "{{ item.review_input_text }}"},
            ],
        },
        "source": {"type": "file_id", "id": uploaded.id},
    },
)
```



### 4. Interpreting the Results

There are three questions to ask when reading a Level 1 benchmark.

First, did the run execute cleanly? If many rows fail to run, the quality metrics are hard to trust.

Second, what is the difference between correctness, usefulness, and noise? These are separate dimensions on purpose. A review can be correct but not useful. It can be useful but too noisy. `overall_pass` is the strict summary metric, while the other three explain why the system passes or fails.

Third, how stable is the benchmark distribution? A large gap between merged and closed-unmerged pull requests can be a signal that the benchmark data itself still has representation problems. That observation motivates Level 2.

## Level 2: Benchmark With Normalized PR Records

Level 1 gives us a working benchmark, but it still feeds the reviewer and graders a fairly raw pull request packet. That can introduce variance. Some pull requests have very large diffs, messy comment history, or ambiguous context, and the evaluator ends up spending too much effort just reconstructing what changed.

Level 2 addresses that by improving the dataset representation rather than changing the benchmark objective.

The new idea is simple: before we ask the reviewer or the graders to reason about a PR, we normalize the PR into a more stable record.

In this example, normalization means:
- deterministic file summaries
- consistent diff trimming
- normalized historical comments
- one cached model-generated `pr_brief`

That `pr_brief` is generated once during dataset preparation and then reused across future runs. The point is not to let another model do the review for us. The point is to make the review task easier to parse consistently.

A useful way to think about Level 2 is:
- Level 1 asks, "can we benchmark this system at all?"
- Level 2 asks, "can we make the benchmark itself more reliable?"

The preparation step looks like this:

```bash
evalcr prepare-dataset --level 2 --cache-key openai_codex
```

Conceptually, each prepared row now has two layers:
1. the raw-ish benchmark evidence: metadata, files, diff, comments
2. the normalized summary layer: `pr_brief`

Then the benchmark run is still the same shape as Level 1:

```bash
evalcr run-evals --level 2 --cache-key openai_codex
```

The reviewer still generates one review per PR. The graders still score correctness, usefulness, noise, and overall pass. What changes is the representation of the pull request that all of those components consume.

### Why Normalization Matters

This is one of the most transferable ideas in eval design. Sometimes the right fix is not a better grader and not a better prompt. Sometimes the right fix is a better task representation.

If Level 1 is noisy, Level 2 gives you a disciplined way to ask whether the noise is coming from the model or from the benchmark format itself.

### Interpreting Level 2

The key comparison is not just whether Level 2 scores are higher. It is whether they are more stable and more interpretable.

Good signs include:
- fewer obviously brittle failures on long or messy pull requests
- clearer grader behavior when the reviewer misses an important issue
- reduced variance across PR subgroups

If Level 2 improves those properties, then it has done its job even before you think about pairwise comparison.


## Level 3: Pairwise Comparison Eval

Once the benchmark input is stable, we can move from absolute scoring to relative comparison.

Pairwise evaluation answers a different question from the benchmark. A benchmark asks, "is this review good enough?" Pairwise asks, "which of these two review policies is better for the same pull request?"

That makes pairwise evaluation especially useful when you are iterating on policy text, instructions, or smaller changes that may not move an absolute pass-rate metric dramatically.

The Level 3 loop looks like this:

```mermaid
flowchart LR
    A[Normalized PR Record] --> B[Generate Baseline Review]
    A --> C[Generate Candidate Review]
    B --> D[Pairwise Judge]
    C --> D
    D -. Choose baseline, candidate, or tie; then move to the next PR .-> A
    classDef reviewer fill:#FEF3C7,stroke:#D97706,color:#78350F,stroke-width:1.5px;
    classDef grader fill:#DCFCE7,stroke:#16A34A,color:#14532D,stroke-width:1.5px;
    class B,C reviewer;
    class D grader;
```

### 1. Prepare Baseline and Candidate Reviews

In the reference implementation, dataset preparation reuses the normalized Level 2 record and then caches two reviews per pull request:
- `baseline_review`
- `candidate_review`

That step is:

```bash
evalcr prepare-dataset --level 3 --cache-key openai_codex
```

This is an important design decision. By the time we reach Level 3, we are no longer comparing policies on top of a noisy raw PR packet. We are comparing them on top of the normalized representation from Level 2.

### 2. Judge the Pair

The Evals run then asks a judge model to choose one of three outcomes:
- `baseline`
- `candidate`
- `tie`

At a conceptual level, the judge sees:
- the normalized PR context
- the baseline review
- the candidate review

and it decides which review is better overall on correctness, usefulness, and noise.

The runnable step is:

```bash
evalcr run-evals --level 3 --cache-key openai_codex
```

### 3. Interpreting Pairwise Results

There are three main outcomes to look for.

If the candidate wins consistently, you have evidence that the candidate policy is an improvement.

If ties dominate, then one of two things is usually true: either the policy change is too small to matter, or the pairwise judge is not discriminating enough.

If the baseline still wins often, that does not necessarily mean the candidate is bad. It means your proposed change is not yet a clear improvement under the current judge and dataset.

### Why Pairwise Comes After Normalization

Pairwise judging is especially sensitive to presentation effects. If the underlying PR record is unstable, the judge may end up preferring whichever review happens to align better with formatting noise rather than whichever policy is actually stronger. That is why this guide introduces normalization before pairwise comparison.

At this point, the three levels form a coherent progression:
1. establish a benchmark
2. stabilize the benchmark inputs
3. compare review policies on the stabilized representation

That sequence gives you a better chance of learning something real from the eval, rather than just measuring noise more precisely.
